In [ ]:
# --- CELL 1: GEOMETRY ---
from ngsolve import *
from netgen.csg import *
import ngsolve.comp.pml as pml
from ngsolve.webgui import Draw

# 1. Define the inner physical domain (the acoustic test region)
# Let's assume these units are in meters (a 2m x 2m x 2m room)
inner_cube = OrthoBrick(Pnt(-1, -1, -1), Pnt(1, 1, 1)).mat("air")

# 2. Define the outer boundary that contains the PML (0.5m thick)
outer_cube = OrthoBrick(Pnt(-1.5, -1.5, -1.5), Pnt(1.5, 1.5, 1.5)).bc("outerbnd")

# 3. The PML region is the geometric difference
pml_region = outer_cube - inner_cube
pml_region.mat("pmlregion")

# 4. Compile the geometry
geo = CSGeometry()
geo.Add(inner_cube)
geo.Add(pml_region)

print("Acoustic domain geometry generated.")

# 5. Visualize (using the quick mesh workaround)
mesh = Mesh(geo.GenerateMesh(maxh=0.5))
Draw(mesh)

Acoustic domain geometry generated.


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [13]:
# --- CELL 2: MESH & PML ---
# 2. Apply the PML mapping to the outer region
mesh.SetPML(pml.BrickRadial(mins=(-1, -1, -1), maxs=(1, 1, 1), alpha=1j), "pmlregion")

print(f"Acoustic mesh ready: {mesh.nv} vertices, {mesh.ne} elements.")

Acoustic mesh ready: 389 vertices, 1476 elements.


In [20]:
# --- CELL 3: ACOUSTIC PHYSICS ---
# 1. Define the complex H1 Space for acoustic pressure
# Order 3 is highly recommended to minimize numerical dispersion of sound waves
fes = H1(mesh, order=4, complex=True, dirichlet="outerbnd")
p, v = fes.TnT()

# 2. Define Acoustic Parameters
c = 343.0          # Speed of sound in air (m/s)
freq = 200.0       # Frequency of the acoustic source in Hz (300 Hz)
omega = 2 * pi * freq
k = omega / c      # Wavenumber

# 3. Assemble the Bilinear Form (Acoustic Helmholtz Equation)
a = BilinearForm(fes)
a += grad(p) * grad(v) * dx - k**2 * p * v * dx
with TaskManager():
    a.Assemble()

# 4. Assemble the Linear Form (Acoustic Monopole Source)
# A tight Gaussian representing a pulsating point source at the center
b = LinearForm(fes)
source_func = exp(-40 * (x**2 + y**2 + z**2))
b += source_func * v * dx
with TaskManager():
    b.Assemble()

print(f"Acoustic system assembled. DOFs: {fes.ndof}")

Acoustic system assembled. DOFs: 17795


In [21]:
# --- CELL 4: SOLVE & DRAW ---
# 1. Create a GridFunction to store the Acoustic Pressure
gfu = GridFunction(fes, name="Pressure")

# 2. Solve the linear system
print("Solving for acoustic pressure field...")
with TaskManager():
    gfu.vec.data = a.mat.Inverse(fes.FreeDofs()) * b.vec
print("Solve complete.")

# 3. Visualize the real part of the acoustic wave
# Use the clipping plane to see the wave radiating from the center
Draw(gfu, mesh, clipping={"z": 0, "y": 0, "x": -1})

Solving for acoustic pressure field...
Solve complete.


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 's…

BaseWebGuiScene